In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion
from collections import deque
from scipy.interpolate import interp1d
from IPython.display import display

# ================= CONFIG =================
NUSCENES_DATAROOT = "/mount/arbeitsdaten/analysis/rasoulta/nuscenes/nuscenes_meta/"
NUSCENES_VERSION = "v1.0-trainval"

FORWARD_MAX = 50.0
BACKWARD_MAX = 10.0
LATERAL_LIMIT = 25.0
FIXED_POINT_COUNT = 50 

# ================= HELPERS =================

def get_yaw(q):
    return Quaternion(q).yaw_pitch_roll[0]

def global_to_local(points_global, ego_translation, ego_rotation):
    points_global = np.array(points_global)
    pts = points_global[:, :2] - np.array(ego_translation[:2])
    yaw = get_yaw(ego_rotation)
    c, s = np.cos(-yaw), np.sin(-yaw)
    rot = np.array([[c,-s],[s,c]])
    local = np.dot(pts, rot.T)
    return np.stack([-local[:,1], local[:,0]], axis=1)

def crop_forward_sector(local_coords):
    if len(local_coords) == 0: return local_coords
    x, y = local_coords[:,0], local_coords[:,1]
    mask = ((y <= FORWARD_MAX) & (y >= -BACKWARD_MAX) & (np.abs(x) <= LATERAL_LIMIT))
    return local_coords[mask]

# ================= INFERENCE & BATCHING =================

def get_reachable_lanes_inference(nmap, ego_trans, ego_rot):
    ego_lane = nmap.get_closest_lane(ego_trans[0], ego_trans[1], radius=5.0)
    if not ego_lane: return []
    
    yaw = get_yaw(ego_rot)
    ego_fwd = np.array([np.cos(yaw), np.sin(yaw)])
    queue, visited, node_path, node_layer = deque([ego_lane]), set(), {}, {}
    
    while queue:
        t = queue.popleft()
        if t in visited: continue
        visited.add(t)
        for layer in ['lane', 'lane_connector']:
            try:
                rec = nmap.get(layer, t)
                pts = nmap.discretize_lanes([t], 0.5)[t]
                node_path[t], node_layer[t] = np.array(pts)[:,:2], layer
                for s in nmap.get_outgoing_lane_ids(t): queue.append(s)
                for adj in [rec.get('left_lane_token'), rec.get('right_lane_token')]:
                    if adj: queue.append(adj)
                break
            except: continue
            
    final = []
    for t, p in node_path.items():
        vec = (p[min(5, len(p)-1)] - p[0])
        vec = vec / (np.linalg.norm(vec) + 1e-6)
        if (node_layer[t] == 'lane' and np.dot(vec, ego_fwd) > -0.2) or node_layer[t] == 'lane_connector':
            final.append((p, node_layer[t], t))
    return final

def get_triplet_batches(nmap, ego_trans, ego_rot, visual_segments):
    inference_map = {token: global_to_local(path, ego_trans, ego_rot) for path, _, token in visual_segments}
    legal_tokens = set(inference_map.keys())

    def get_min_dist(token):
        return np.min(np.linalg.norm(inference_map[token], axis=1))

    roots = sorted([t for t in legal_tokens if get_min_dist(t) <= 5.0 and np.max(inference_map[t][:,1]) > 0], 
                   key=get_min_dist)[:5]

    def walk(curr, chain, can_branch=True):
        curr_local = inference_map[curr]
        if curr_local[-1, 1] >= FORWARD_MAX or len(chain) >= 5: return [chain]
        successors = nmap.get_outgoing_lane_ids(curr)
        valid = sorted([s for s in successors if s in legal_tokens and s not in chain], 
                       key=lambda s: np.linalg.norm(inference_map[s][0]))
        if not valid: return [chain]
        res = []
        for i, s in enumerate(valid):
            if np.linalg.norm(curr_local[-1] - inference_map[s][0]) < 8.0:
                res.extend(walk(s, chain + [s], can_branch if i == 0 else False))
        return res if res else [chain]

    batches = []
    seen = set()
    for r in roots:
        for path in walk(r, [r]):
            vis = [t for t in path if np.any((inference_map[t][:,1] >= -BACKWARD_MAX) & (inference_map[t][:,1] <= FORWARD_MAX))]
            if vis and tuple(vis) not in seen:
                stitched = np.vstack([inference_map[t] for t in vis])
                batches.append((crop_forward_sector(stitched), tuple(vis)))
                seen.add(tuple(vis))
    return batches

def calculate_feature_matrix(local_points, target_points=50):
    diffs = np.diff(local_points, axis=0)
    dists = np.linalg.norm(diffs, axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0)
    _, unique_indices = np.unique(cum_dist, return_index=True)
    if len(unique_indices) < 3: return None
    
    local_points = local_points[unique_indices]
    cum_dist = cum_dist[unique_indices]
    total_len = cum_dist[-1]
    if total_len < 1.0: return None

    new_dists = np.linspace(0, total_len, target_points)
    f_interp = interp1d(cum_dist, local_points, kind='linear', axis=0)
    new_points = f_interp(new_dists)

    grads = np.gradient(new_points, axis=0)
    norms = np.linalg.norm(grads, axis=1)[:, np.newaxis]
    norms[norms == 0] = 1.0
    u = grads / norms
    
    d2_grads = np.gradient(grads, axis=0)
    k = (grads[:,0]*d2_grads[:,1] - grads[:,1]*d2_grads[:,0]) / (np.power(norms[:,0], 3) + 1e-6)
    
    return np.stack([new_points[:,0], new_points[:,1], u[:,0], u[:,1], np.clip(k, -0.5, 0.5), new_dists/total_len, np.zeros(target_points)], axis=1)

# ================= DISPLAY ENGINE =================

def render_detailed_map(ax, nmap, ego_trans, ego_rot):
    layers = ['lane', 'road_segment', 'ped_crossing', 'walkway', 'carpark_area', 'stop_line']
    colors = {'lane': '#222222', 'road_segment': '#151515', 'ped_crossing': '#333333', 
              'walkway': '#1a1a1a', 'carpark_area': '#121212', 'stop_line': '#444400'}
    recs = nmap.get_records_in_radius(ego_trans[0], ego_trans[1], FORWARD_MAX, layers + ['lane_divider', 'road_divider'])
    for lyr in layers:
        for t in recs.get(lyr, []):
            rec = nmap.get(lyr, t)
            if 'polygon_token' not in rec: continue
            poly = nmap.extract_polygon(rec['polygon_token'])
            coords = crop_forward_sector(global_to_local(np.array(poly.exterior.coords)[:,:2], ego_trans, ego_rot))
            if len(coords) >= 3: ax.add_patch(MplPolygon(coords, facecolor=colors[lyr], zorder=1, alpha=0.8))
    for lyr, col, lw in [('road_divider', '#FF8C00', 2.5), ('lane_divider', '#FFFFFF', 1.2)]:
        for t in recs.get(lyr, []):
            line = nmap.extract_line(nmap.get(lyr, t)['line_token'])
            coords = crop_forward_sector(global_to_local(np.array(line.coords)[:,:2], ego_trans, ego_rot))
            if len(coords) >= 2: ax.plot(coords[:,0], coords[:,1], color=col, linewidth=lw, alpha=0.7, zorder=2)

def display_full_inference(nmap, ego_trans, ego_rot, visual_segments):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9), facecolor='black')
    plt.subplots_adjust(wspace=0.05)
    for ax in [ax1, ax2]:
        ax.set_facecolor('black')
        ax.plot(0, 0, 'r^', markersize=15, zorder=30)
    for p_global, layer, t in visual_segments:
        local = crop_forward_sector(global_to_local(p_global, ego_trans, ego_rot))
        if len(local) < 2: continue
        color = 'cyan' if layer == 'lane_connector' else 'magenta'
        ax1.plot(local[:,0], local[:,1], color=color, lw=3, alpha=0.8)
        m = len(local) // 2
        ax1.text(local[m, 0] + 1.2, local[m, 1] + 1.2, t[:4], color='white', fontsize=10, 
                 weight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor=color, pad=1))
    render_detailed_map(ax2, nmap, ego_trans, ego_rot)
    ax1.set_title("OVERVIEW: ALL REACHABLE SEGMENTS", color='white', loc='left', fontsize=14)
    ax2.set_title("HD MAP GROUND TRUTH (DETAIL)", color='white', loc='right', fontsize=14)
    for ax in [ax1, ax2]:
        ax.set_xlim(-LATERAL_LIMIT, LATERAL_LIMIT); ax.set_ylim(-BACKWARD_MAX, FORWARD_MAX); ax.axis('off')
    plt.show()

def display_batch_results(nmap, ego_trans, ego_rot, batches, visual_segments):
    for idx, (path_geom, batch_ids) in enumerate(batches):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9), facecolor='black')
        plt.subplots_adjust(wspace=0.05)
        for ax in [ax1, ax2]:
            ax.set_facecolor('black')
            ax.plot(0, 0, 'r^', markersize=15, zorder=30)
        for p_global, layer, t in visual_segments:
            local = crop_forward_sector(global_to_local(p_global, ego_trans, ego_rot))
            if len(local) < 2: continue
            is_in_batch = t in batch_ids
            color = '#00FF00' if is_in_batch else '#2A2A2A'
            ax1.plot(local[:,0], local[:,1], color=color, lw=6 if is_in_batch else 2, zorder=20 if is_in_batch else 1)
            if is_in_batch:
                m = len(local) // 2
                ax1.text(local[m, 0] + 2.5, local[m, 1] + 2.5, t[:4], color='white', fontsize=12, 
                         weight='bold', bbox=dict(facecolor='black', alpha=0.8, edgecolor='#00FF00', pad=3))
        render_detailed_map(ax2, nmap, ego_trans, ego_rot)
        ax1.set_title(f"BATCH {idx}: {' → '.join([t[:4] for t in batch_ids])}", color='#00FF00', loc='left', fontsize=14)
        ax2.set_title("DETAILED MAP (CLEAN)", color='white', loc='right', fontsize=14)
        for ax in [ax1, ax2]:
            ax.set_xlim(-LATERAL_LIMIT, LATERAL_LIMIT); ax.set_ylim(-BACKWARD_MAX, FORWARD_MAX); ax.axis('off')
        plt.show()

# ================= EXECUTION =================

if __name__ == "__main__":
    nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=False)
    sample_token = random.choice([s['token'] for s in nusc.sample])
    sample = nusc.get('sample', sample_token)
    log = nusc.get('log', nusc.get('scene', sample['scene_token'])['log_token'])
    nmap = NuScenesMap(dataroot=NUSCENES_DATAROOT, map_name=log['location'])
    ego_pose = nusc.get('ego_pose', nusc.get('sample_data', sample['data']['LIDAR_TOP'])['ego_pose_token'])
    ego_trans, ego_rot = ego_pose['translation'], ego_pose['rotation']

    # --- 1. RUN INFERENCE ---
    visual_segs = get_reachable_lanes_inference(nmap, ego_trans, ego_rot)
    final_batches = get_triplet_batches(nmap, ego_trans, ego_rot, visual_segs)
    
    # --- 2. PRINT BATCHING CONSOLE OUTPUT ---
    print(f"Sample: {sample_token}")
    print("\n" + "="*50)
    print("=== TENSOR COLUMN LEGEND ===")
    print("Col 0: Lateral (m)   - Sideways distance")
    print("Col 1: Forward (m)   - Distance ahead")
    print("Col 2: Dir_Lat       - Unit vector X (heading)")
    print("Col 3: Dir_Fwd       - Unit vector Y (heading)")
    print("Col 4: Curvature     - Turn sharpness (-0.5 to 0.5)")
    print("Col 5: s (0.0-1.0)   - Progress along path")
    print("Col 6: Flag          - (Currently Unused)")
    print("="*50 + "\n")
    
    print(f"=== IDENTIFIED DRIVABLE PATHS (Cropped to {FORWARD_MAX}m) ===")
    
    encoded_features = []
    for i, (path, ids) in enumerate(final_batches):
        if len(path) > 3:
            mat = calculate_feature_matrix(path, target_points=FIXED_POINT_COUNT)
            if mat is not None:
                encoded_features.append(mat)
                chain_str = " -> ".join([t[:4] for t in ids])
                print(f"Batch {i}: [{chain_str}] (Length: {path[-1,1]:.1f}m)")

    if encoded_features:
        print(f"\nFinal Input Tensor Shape: {np.array(encoded_features).shape} (Batch, Points, Features)")
    else:
        print("No valid paths found.")

    # --- 3. RUN VISUALIZATIONS ---
    if visual_segs:
        display_full_inference(nmap, ego_trans, ego_rot, visual_segs)
    if final_batches:
        display_batch_results(nmap, ego_trans, ego_rot, final_batches, visual_segs)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion
from collections import deque
from scipy.interpolate import interp1d
from IPython.display import display, Image

# ================= CONFIG =================
NUSCENES_DATAROOT = "/mount/arbeitsdaten/analysis/rasoulta/nuscenes/nuscenes_meta/"
NUSCENES_VERSION = "v1.0-trainval"

FORWARD_MAX = 50.0
BACKWARD_MAX = 10.0
LATERAL_LIMIT = 25.0
FIXED_POINT_COUNT = 50 

# ================= HELPERS =================

def get_yaw(q):
    return Quaternion(q).yaw_pitch_roll[0]

def global_to_local(points_global, ego_translation, ego_rotation):
    points_global = np.array(points_global)
    pts = points_global[:, :2] - np.array(ego_translation[:2])
    yaw = get_yaw(ego_rotation)
    c, s = np.cos(-yaw), np.sin(-yaw)
    rot = np.array([[c,-s],[s,c]])
    local = np.dot(pts, rot.T)
    return np.stack([-local[:,1], local[:,0]], axis=1)

def get_safe_record(nmap, token):
    for layer in ['lane','lane_connector']:
        try: return nmap.get(layer, token), layer
        except: continue
    return None, None

def crop_forward_sector(local_coords):
    if len(local_coords) == 0: return local_coords
    x, y = local_coords[:,0], local_coords[:,1]
    mask = ((y <= FORWARD_MAX) & (y >= -BACKWARD_MAX) & (np.abs(x) <= LATERAL_LIMIT))
    return local_coords[mask]

def lane_direction_valid(path, ego_forward):
    if len(path) < 6: return True
    lane_vec = (path[5] - path[0]) / (np.linalg.norm(path[5] - path[0]) + 1e-6)
    return np.dot(lane_vec, ego_forward) > -0.2

# ================= BATCHING LOGIC (STRICT FLOW) =================

# def get_triplet_batches(nmap, ego_trans, ego_rot):
#     # 1. Source of Truth from Inference
#     visual_segments = get_reachable_lanes(nmap, ego_trans, ego_rot)
#     if not visual_segments: return []

#     # Map for geometry lookup
#     inference_map = {token: global_to_local(path, ego_trans, ego_rot) for path, _, token in visual_segments}
#     legal_tokens = set(inference_map.keys())

#     # Helper: Distance based on midpoint
#     def get_midpoint_dist(token):
#         pts = inference_map[token]
#         mid_pt = pts[len(pts)//2]
#         return np.linalg.norm(mid_pt)

#     # Filter for segments whose midpoints are parallel/ahead
#     candidate_list = []
#     for t in legal_tokens:
#         pts = inference_map[t]
#         mid_pt = pts[len(pts)//2]
#         if mid_pt[1] > -2.0: 
#             candidate_list.append((t, get_midpoint_dist(t)))

#     # Sort candidates by distance (Nearest first)
#     sorted_candidates = [c[0] for c in sorted(candidate_list, key=lambda x: x[1])]

#     # Tiers
#     tier_1_roots = sorted_candidates[:5]
#     higher_tier_roots = sorted_candidates[5:15] # Expand search for higher IDs

#     # 2. STITCHING ENGINE
#     def walk_branches(current_token, current_chain, allow_branching=True):
#         current_local = inference_map[current_token]
#         # Stop if we hit 50m
#         if current_local[-1, 1] >= FORWARD_MAX:
#             return [current_chain]

#         successors = nmap.get_outgoing_lane_ids(current_token)
#         valid_successors = [s for s in successors if s in legal_tokens and s not in current_chain]

#         if not valid_successors:
#             return [current_chain]

#         # Sort successors by proximity
#         valid_successors = sorted(valid_successors, key=lambda s: get_midpoint_dist(s))

#         results = []
#         for i, succ in enumerate(valid_successors):
#             succ_local = inference_map[succ]
#             dist = np.linalg.norm(current_local[-1] - succ_local[0])
            
#             if dist < 8.0 and succ_local[-1, 1] > current_local[-1, 1]:
#                 # "Second place" logic: Build and stop further branching if i > 0
#                 # i=0 is the primary "best" path
#                 next_branching = allow_branching if i == 0 else False
#                 results.extend(walk_branches(succ, current_chain + [succ], next_branching))
        
#         return results if results else [current_chain]

#     # 3. EXECUTE & DISMISS OUT-OF-BOUNDS IDs
#     batches = []
#     processed_fingerprints = set()

#     for root_list in [tier_1_roots, higher_tier_roots]:
#         for root in root_list:
#             paths = walk_branches(root, [root])
#             for p in paths:
#                 # FILTER IDs: Dismiss any ID in the chain that is outside the 50m radius
#                 visible_ids = []
#                 for t in p:
#                     pts = inference_map[t]
#                     # Check if ANY point of the segment is within bounds
#                     # (Y: -10 to 50, X: -25 to 25)
#                     mask = (pts[:,1] <= FORWARD_MAX) & (pts[:,1] >= -BACKWARD_MAX) & (np.abs(pts[:,0]) <= LATERAL_LIMIT)
#                     if np.any(mask):
#                         visible_ids.append(t)
                
#                 if not visible_ids: continue
                
#                 fprint = tuple(visible_ids)
#                 if fprint not in processed_fingerprints:
#                     # Stitch only the relevant parts of the path
#                     stitched_geom = np.vstack([inference_map[t] for t in visible_ids])
#                     final_cropped = crop_forward_sector(stitched_geom)
                    
#                     if len(final_cropped) > 5:
#                         batches.append((final_cropped, fprint))
#                         processed_fingerprints.add(fprint)

#     return batches

def get_triplet_batches(nmap, ego_trans, ego_rot):
    # 1. Source of Truth: Exact segments from the visualization
    visual_segments = get_reachable_lanes(nmap, ego_trans, ego_rot)
    if not visual_segments: 
        return []

    # Local geometry lookup
    inference_map = {token: global_to_local(path, ego_trans, ego_rot) for path, _, token in visual_segments}
    legal_tokens = set(inference_map.keys())

    # Helper: Find the point in the segment closest to the ego (0,0)
    def get_min_dist_to_ego(token):
        pts = inference_map[token]
        # Calculate distance from (0,0) to every point in the segment
        dists = np.linalg.norm(pts, axis=1)
        return np.min(dists)

    # 2. SELECTION: Find IDs where ANY part is within 5m of ego
    candidates = []
    for t in legal_tokens:
        min_dist = get_min_dist_to_ego(t)
        pts = inference_map[t]
        
        # Rule: Any part of the segment is within 5m AND it's not entirely behind us
        if min_dist <= 5.0 and np.max(pts[:, 1]) > 0:
            candidates.append((t, min_dist))

    # Sort by closest point proximity
    sorted_candidates = [c[0] for c in sorted(candidates, key=lambda x: x[1])]
    
    # Priority Tiers
    tier_1_roots = sorted_candidates[:5]
    higher_tier_roots = sorted_candidates[5:]

    # 3. STITCHING ENGINE
    def walk_branches(current_token, current_chain, allow_branching=True):
        current_local = inference_map[current_token]
        
        # Stop at 50m horizon or max segments
        if current_local[-1, 1] >= FORWARD_MAX or len(current_chain) >= 5:
            return [current_chain]

        successors = nmap.get_outgoing_lane_ids(current_token)
        valid_successors = [s for s in successors if s in legal_tokens and s not in current_chain]

        if not valid_successors:
            return [current_chain]

        # Sort successors by their entry point distance to the current segment's end
        valid_successors = sorted(valid_successors, key=lambda s: np.linalg.norm(inference_map[s][0]))

        results = []
        for i, succ in enumerate(valid_successors):
            succ_local = inference_map[succ]
            # Verify physical connection
            dist = np.linalg.norm(current_local[-1] - succ_local[0])
            
            if dist < 8.0 and succ_local[-1, 1] > current_local[-1, 1]:
                # Primary path (i=0) continues branching; secondary stops after one stitch
                next_branching = allow_branching if i == 0 else False
                results.extend(walk_branches(succ, current_chain + [succ], next_branching))
        
        return results if results else [current_chain]

    # 4. EXECUTE AND SPATIAL PURGE
    batches = []
    processed_fingerprints = set()

    for root_group in [tier_1_roots, higher_tier_roots]:
        for root in root_group:
            paths = walk_branches(root, [root])
            for p in paths:
                # DISMISS IDs that are strictly outside the 50m radius
                visible_ids = []
                for t in p:
                    pts = inference_map[t]
                    # Check if the segment exists at all within our forward 50m window
                    if np.any((pts[:, 1] >= -BACKWARD_MAX) & (pts[:, 1] <= FORWARD_MAX)):
                        visible_ids.append(t)
                
                if not visible_ids: continue
                
                fprint = tuple(visible_ids)
                if fprint not in processed_fingerprints:
                    stitched_pts = np.vstack([inference_map[t] for t in visible_ids])
                    cropped = crop_forward_sector(stitched_pts)
                    
                    if len(cropped) > 5:
                        batches.append((cropped, fprint))
                        processed_fingerprints.add(fprint)

    return batches

def calculate_feature_matrix(pts, target=50):
    diffs = np.diff(pts, axis=0)
    dists = np.linalg.norm(diffs, axis=1)
    cum = np.insert(np.cumsum(dists), 0, 0)
    _, idx = np.unique(cum, return_index=True)
    if len(idx) < 3: return None
    pts, cum = pts[idx], cum[idx]
    if cum[-1] < 1.0: return None
    new_d = np.linspace(0, cum[-1], target)
    interp = interp1d(cum, pts, axis=0)(new_d)
    g = np.gradient(interp, axis=0)
    n = np.linalg.norm(g, axis=1)[:, None]
    n[n == 0] = 1.0
    u = g / n
    g2 = np.gradient(g, axis=0)
    k = (g[:,0]*g2[:,1] - g[:,1]*g2[:,0]) / (np.power(n[:,0], 3) + 1e-6)
    return np.stack([interp[:,0], interp[:,1], u[:,0], u[:,1], np.clip(k, -0.5, 0.5), new_d/cum[-1], np.zeros(target)], axis=1)

# ================= VISUALIZATION LOGIC =================

def get_reachable_lanes(nmap, ego_trans, ego_rot):
    ego_lane = nmap.get_closest_lane(ego_trans[0], ego_trans[1], radius=5.0)
    if not ego_lane:
        recs = nmap.get_records_in_radius(ego_trans[0], ego_trans[1], 20.0, ['lane'])
        if recs['lane']: ego_lane = recs['lane'][0]
        else: return []

    yaw = get_yaw(ego_rot)
    ego_forward = np.array([np.cos(yaw), np.sin(yaw)])
    queue = deque([ego_lane])
    visited, node_path, node_layer = set(), {}, {}

    while queue:
        token = queue.popleft()
        if token in visited: continue
        visited.add(token)
        rec, layer = get_safe_record(nmap, token)
        if rec is None: continue
        out = nmap.discretize_lanes([token], 0.5)
        pts = out.get(token, []) if isinstance(out, dict) else out[0]
        if len(pts) > 0:
            node_path[token], node_layer[token] = np.array(pts)[:,:2], layer
        for s in nmap.get_outgoing_lane_ids(token): queue.append(s)
        for key in ['left_lane_token', 'right_lane_token']:
            if rec.get(key): queue.append(rec[key])

    legal_tokens = {t for t, p in node_path.items() if node_layer[t] == 'lane' and lane_direction_valid(p, ego_forward)}
    final_segments = []
    for t, p in node_path.items():
        layer = node_layer[t]
        if (layer == 'lane' and t in legal_tokens) or (layer == 'lane_connector' and any(s in legal_tokens for s in nmap.get_outgoing_lane_ids(t))):
            final_segments.append((p, layer, t))
    return final_segments

# ================= MAIN =================

if __name__ == "__main__":
    # nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=False)
    sample_token = random.choice([s['token'] for s in nusc.sample])
    sample = nusc.get('sample', sample_token)
    log = nusc.get('log', nusc.get('scene', sample['scene_token'])['log_token'])
    nmap = NuScenesMap(dataroot=NUSCENES_DATAROOT, map_name=log['location'])
    ego_pose = nusc.get('ego_pose', nusc.get('sample_data', sample['data']['LIDAR_TOP'])['ego_pose_token'])
    ego_trans, ego_rot = ego_pose['translation'], ego_pose['rotation']

    # 1. BATCHING
    batches = get_triplet_batches(nmap, ego_trans, ego_rot)
    
    print(f"Sample: {sample_token}")
    print("\n" + "="*50)
    print("=== TENSOR COLUMN LEGEND ===")
    print("0:Lat, 1:Fwd, 2:U_Lat, 3:U_Fwd, 4:Kappa, 5:s, 6:Flag")
    print("="*50 + "\n")
    
    print(f"=== IDENTIFIED DRIVABLE PATHS (Cropped to {FORWARD_MAX}m) ===")
    
    encoded_features = []
    for i, (path, ids) in enumerate(batches):
        mat = calculate_feature_matrix(path, target=FIXED_POINT_COUNT)
        if mat is not None:
            encoded_features.append(mat)
            print(f"Batch {i}: [{' -> '.join([t[:4] for t in ids])}] (Len: {path[-1,1]:.1f}m)")

    if encoded_features:
        print(f"\nFinal Input Tensor Shape: {np.array(encoded_features).shape}")
    else:
        print("No batches found for this sample.")

    # 2. VISUALIZATION
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), facecolor='black')
    for ax in [ax1, ax2]:
        ax.set_facecolor('black')
        ax.plot(0, 0, 'r^', markersize=20)

    centerlines = get_reachable_lanes(nmap, ego_trans, ego_rot)
    for p, layer, t in centerlines:
        local = crop_forward_sector(global_to_local(p, ego_trans, ego_rot))
        if len(local) >= 2:
            ax1.plot(local[:,0], local[:,1], color='cyan' if layer=='lane_connector' else 'magenta', lw=4, alpha=0.8)
            ax1.text(local[len(local)//2, 0], local[len(local)//2, 1], t[:4], color='white', weight='bold')

    recs = nmap.get_records_in_radius(ego_trans[0], ego_trans[1], FORWARD_MAX, ['road_segment','lane','ped_crossing'])
    for lyr, col in {'road_segment':'#1f1f1f', 'lane':'#3d3d3d', 'ped_crossing':'#6b6b6b'}.items():
        for t in recs.get(lyr, []):
            poly = nmap.extract_polygon(nmap.get(lyr, t)['polygon_token'])
            coords = crop_forward_sector(global_to_local(np.array(poly.exterior.coords)[:,:2], ego_trans, ego_rot))
            if len(coords) >= 3: ax2.add_patch(MplPolygon(coords, facecolor=col, alpha=0.6))

    for ax in [ax1, ax2]:
        ax.set_xlim(-LATERAL_LIMIT, LATERAL_LIMIT); ax.set_ylim(-BACKWARD_MAX, FORWARD_MAX); ax.axis('off')

    plt.savefig("printed_batches_viz.png", facecolor='black', bbox_inches='tight', dpi=150)
    plt.close(); display(Image(filename="printed_batches_viz.png"))

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap
from pyquaternion import Quaternion
from collections import deque
from scipy.interpolate import interp1d
from IPython.display import display

# ================= CONFIG =================
NUSCENES_DATAROOT = "/mount/arbeitsdaten/analysis/rasoulta/nuscenes/nuscenes_meta/"
NUSCENES_VERSION = "v1.0-trainval"

FORWARD_MAX = 50.0
BACKWARD_MAX = 10.0
LATERAL_LIMIT = 25.0
FIXED_POINT_COUNT = 50 

# ================= HELPERS =================

def get_yaw(q):
    return Quaternion(q).yaw_pitch_roll[0]

def global_to_local(points_global, ego_translation, ego_rotation):
    points_global = np.array(points_global)
    pts = points_global[:, :2] - np.array(ego_translation[:2])
    yaw = get_yaw(ego_rotation)
    c, s = np.cos(-yaw), np.sin(-yaw)
    rot = np.array([[c,-s],[s,c]])
    local = np.dot(pts, rot.T)
    return np.stack([-local[:,1], local[:,0]], axis=1)

def crop_forward_sector(local_coords):
    if len(local_coords) == 0: return local_coords
    x, y = local_coords[:,0], local_coords[:,1]
    mask = ((y <= FORWARD_MAX) & (y >= -BACKWARD_MAX) & (np.abs(x) <= LATERAL_LIMIT))
    return local_coords[mask]

# ================= INFERENCE & BATCHING =================

def get_reachable_lanes_inference(nmap, ego_trans, ego_rot):
    ego_lane = nmap.get_closest_lane(ego_trans[0], ego_trans[1], radius=5.0)
    if not ego_lane: return []
    
    yaw = get_yaw(ego_rot)
    ego_fwd = np.array([np.cos(yaw), np.sin(yaw)])
    queue, visited, node_path, node_layer = deque([ego_lane]), set(), {}, {}
    
    while queue:
        t = queue.popleft()
        if t in visited: continue
        visited.add(t)
        for layer in ['lane', 'lane_connector']:
            try:
                rec = nmap.get(layer, t)
                pts = nmap.discretize_lanes([t], 0.5)[t]
                node_path[t], node_layer[t] = np.array(pts)[:,:2], layer
                for s in nmap.get_outgoing_lane_ids(t): queue.append(s)
                for adj in [rec.get('left_lane_token'), rec.get('right_lane_token')]:
                    if adj: queue.append(adj)
                break
            except: continue
            
    final = []
    for t, p in node_path.items():
        vec = (p[min(5, len(p)-1)] - p[0])
        vec = vec / (np.linalg.norm(vec) + 1e-6)
        if (node_layer[t] == 'lane' and np.dot(vec, ego_fwd) > -0.2) or node_layer[t] == 'lane_connector':
            final.append((p, node_layer[t], t))
    return final

def get_triplet_batches(nmap, ego_trans, ego_rot, visual_segments):
    inference_map = {token: global_to_local(path, ego_trans, ego_rot) for path, _, token in visual_segments}
    legal_tokens = set(inference_map.keys())

    def get_min_dist(token):
        return np.min(np.linalg.norm(inference_map[token], axis=1))

    roots = sorted([t for t in legal_tokens if get_min_dist(t) <= 5.0 and np.max(inference_map[t][:,1]) > 0], 
                   key=get_min_dist)[:5]

    def walk(curr, chain, can_branch=True):
        curr_local = inference_map[curr]
        if curr_local[-1, 1] >= FORWARD_MAX or len(chain) >= 5: return [chain]
        successors = nmap.get_outgoing_lane_ids(curr)
        valid = sorted([s for s in successors if s in legal_tokens and s not in chain], 
                       key=lambda s: np.linalg.norm(inference_map[s][0]))
        if not valid: return [chain]
        res = []
        for i, s in enumerate(valid):
            if np.linalg.norm(curr_local[-1] - inference_map[s][0]) < 8.0:
                res.extend(walk(s, chain + [s], can_branch if i == 0 else False))
        return res if res else [chain]

    batches = []
    seen = set()
    for r in roots:
        for path in walk(r, [r]):
            vis = [t for t in path if np.any((inference_map[t][:,1] >= -BACKWARD_MAX) & (inference_map[t][:,1] <= FORWARD_MAX))]
            if vis and tuple(vis) not in seen:
                stitched = np.vstack([inference_map[t] for t in vis])
                batches.append((crop_forward_sector(stitched), tuple(vis)))
                seen.add(tuple(vis))
    return batches

def calculate_feature_matrix(local_points, target_points=50):
    # 1. REMOVE DUPLICATES & CALCULATE DISTANCES
    # Stitching often results in duplicate points at the boundary (end of ID1 == start of ID2)
    diffs = np.diff(local_points, axis=0)
    dists = np.linalg.norm(diffs, axis=1)
    cum_dist = np.insert(np.cumsum(dists), 0, 0)
    
    # Strictly filter for unique cumulative distances to prevent interp1d crashes
    _, unique_indices = np.unique(cum_dist, return_index=True)
    if len(unique_indices) < 3: return None
    
    local_points = local_points[unique_indices]
    cum_dist = cum_dist[unique_indices]
    total_len = cum_dist[-1]
    
    # If the path is too short (e.g., < 1m), it's not useful for classification
    if total_len < 1.0: return None

    # 2. RESAMPLE (Ensures exactly 50 points regardless of lane segment length)
    new_dists = np.linspace(0, total_len, target_points)
    f_interp = interp1d(cum_dist, local_points, kind='linear', axis=0)
    new_points = f_interp(new_dists)

    # 3. CALCULATE HEADING (Unit Vectors)
    grads = np.gradient(new_points, axis=0)
    norms = np.linalg.norm(grads, axis=1)[:, np.newaxis]
    norms[norms == 0] = 1.0
    u = grads / norms
    
    # 4. CALCULATE CURVATURE (Turn Intensity)
    d2_grads = np.gradient(grads, axis=0)
    # Using the standard 2D curvature formula
    k = (grads[:,0]*d2_grads[:,1] - grads[:,1]*d2_grads[:,0]) / (np.power(norms[:,0], 3) + 1e-6)
    
    # 5. PROGRESS VECTOR (Critical for labeling your 2 PNGs)
    # s starts at 0.0 (ego) and goes to 1.0 (50m out)
    s_vec = new_dists / total_len 
    
    # 6. RETURN ENCODED TENSOR [50, 7]
    return np.stack([
        new_points[:,0],      # 0: Lateral (X)
        new_points[:,1],      # 1: Forward (Y)
        u[:,0],               # 2: Dir Lat
        u[:,1],               # 3: Dir Fwd
        np.clip(k, -0.5, 0.5),# 4: Curvature
        s_vec,                # 5: Progress (s)
        np.zeros(target_points) # 6: Flag (Padding)
    ], axis=1)

# ================= DISPLAY ENGINE =================

def render_detailed_map(ax, nmap, ego_trans, ego_rot):
    layers = ['lane', 'road_segment', 'ped_crossing', 'walkway', 'carpark_area', 'stop_line']
    colors = {'lane': '#222222', 'road_segment': '#151515', 'ped_crossing': '#333333', 
              'walkway': '#1a1a1a', 'carpark_area': '#121212', 'stop_line': '#444400'}
    recs = nmap.get_records_in_radius(ego_trans[0], ego_trans[1], FORWARD_MAX, layers + ['lane_divider', 'road_divider'])
    for lyr in layers:
        for t in recs.get(lyr, []):
            rec = nmap.get(lyr, t)
            if 'polygon_token' not in rec: continue
            poly = nmap.extract_polygon(rec['polygon_token'])
            coords = crop_forward_sector(global_to_local(np.array(poly.exterior.coords)[:,:2], ego_trans, ego_rot))
            if len(coords) >= 3: ax.add_patch(MplPolygon(coords, facecolor=colors[lyr], zorder=1, alpha=0.8))
    for lyr, col, lw in [('road_divider', '#FF8C00', 2.5), ('lane_divider', '#FFFFFF', 1.2)]:
        for t in recs.get(lyr, []):
            line = nmap.extract_line(nmap.get(lyr, t)['line_token'])
            coords = crop_forward_sector(global_to_local(np.array(line.coords)[:,:2], ego_trans, ego_rot))
            if len(coords) >= 2: ax.plot(coords[:,0], coords[:,1], color=col, linewidth=lw, alpha=0.7, zorder=2)

def display_full_inference(nmap, ego_trans, ego_rot, visual_segments):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9), facecolor='black')
    plt.subplots_adjust(wspace=0.05)
    for ax in [ax1, ax2]:
        ax.set_facecolor('black')
        ax.plot(0, 0, 'r^', markersize=15, zorder=30)
    for p_global, layer, t in visual_segments:
        local = crop_forward_sector(global_to_local(p_global, ego_trans, ego_rot))
        if len(local) < 2: continue
        color = 'cyan' if layer == 'lane_connector' else 'magenta'
        ax1.plot(local[:,0], local[:,1], color=color, lw=3, alpha=0.8)
        m = len(local) // 2
        ax1.text(local[m, 0] + 1.2, local[m, 1] + 1.2, t[:4], color='white', fontsize=10, 
                 weight='bold', bbox=dict(facecolor='black', alpha=0.6, edgecolor=color, pad=1))
    render_detailed_map(ax2, nmap, ego_trans, ego_rot)
    ax1.set_title("OVERVIEW: ALL REACHABLE SEGMENTS", color='white', loc='left', fontsize=14)
    ax2.set_title("HD MAP GROUND TRUTH (DETAIL)", color='white', loc='right', fontsize=14)
    for ax in [ax1, ax2]:
        ax.set_xlim(-LATERAL_LIMIT, LATERAL_LIMIT); ax.set_ylim(-BACKWARD_MAX, FORWARD_MAX); ax.axis('off')
    plt.show()

def display_batch_results(nmap, ego_trans, ego_rot, batches, visual_segments):
    for idx, (path_geom, batch_ids) in enumerate(batches):
        # Determine how many images to show for this batch
        # If we have [ID0, ID1, ID2], we show two stages: (ID0->ID1) and (ID1->ID2)
        num_stages = 1
        if len(batch_ids) >= 3:
            num_stages = 2
            
        for stage in range(num_stages):
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9), facecolor='black')
            plt.subplots_adjust(wspace=0.05)
            
            for ax in [ax1, ax2]:
                ax.set_facecolor('black')
                ax.plot(0, 0, 'r^', markersize=15, zorder=30)

            # Define which IDs are highlighted in this specific stage
            if num_stages == 2:
                # Stage 0 highlights ID 0 and 1; Stage 1 highlights ID 1 and 2
                active_ids = [batch_ids[stage], batch_ids[stage + 1]]
                stage_title = f" (Stage {stage + 1}: {active_ids[0][:4]} -> {active_ids[1][:4]})"
            else:
                active_ids = list(batch_ids)
                stage_title = ""

            for p_global, layer, t in visual_segments:
                local = crop_forward_sector(global_to_local(p_global, ego_trans, ego_rot))
                if len(local) < 2: continue
                
                is_in_active_pair = t in active_ids
                # Dim other segments in the batch that aren't part of the current active pair
                is_in_total_batch = t in batch_ids
                
                if is_in_active_pair:
                    color = '#00FF00' # Bright Green
                    lw = 6
                    alpha = 1.0
                    z = 25
                elif is_in_total_batch:
                    color = '#004400' # Darker Green for context within batch
                    lw = 3
                    alpha = 0.6
                    z = 20
                else:
                    color = '#2A2A2A' # Dim Grey for everything else
                    lw = 2
                    alpha = 0.4
                    z = 1
                
                ax1.plot(local[:,0], local[:,1], color=color, lw=lw, alpha=alpha, zorder=z)
                
                if is_in_active_pair:
                    m = len(local) // 2
                    # Offset labels based on their order in the active pair to prevent overlaps
                    offset = 2.5 if t == active_ids[0] else 5.0
                    ax1.text(local[m, 0] + offset, local[m, 1] + offset, t[:4], color='white', 
                             fontsize=12, weight='bold', 
                             bbox=dict(facecolor='black', alpha=0.8, edgecolor='#00FF00', pad=3))

            render_detailed_map(ax2, nmap, ego_trans, ego_rot)
            
            chain_str = " → ".join([t[:4] for t in batch_ids])
            ax1.set_title(f"BATCH {idx}: [{chain_str}]{stage_title}", color='#00FF00', loc='left', fontsize=14)
            ax2.set_title("HD MAP GROUND TRUTH (DETAIL)", color='white', loc='right', fontsize=14)

            for ax in [ax1, ax2]:
                ax.set_xlim(-LATERAL_LIMIT, LATERAL_LIMIT)
                ax.set_ylim(-BACKWARD_MAX, FORWARD_MAX)
                ax.axis('off')
            plt.show()

# ================= EXECUTION =================

if __name__ == "__main__":
    # nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_DATAROOT, verbose=False)
    sample_token = random.choice([s['token'] for s in nusc.sample])
    sample = nusc.get('sample', sample_token)
    log = nusc.get('log', nusc.get('scene', sample['scene_token'])['log_token'])
    nmap = NuScenesMap(dataroot=NUSCENES_DATAROOT, map_name=log['location'])
    ego_pose = nusc.get('ego_pose', nusc.get('sample_data', sample['data']['LIDAR_TOP'])['ego_pose_token'])
    ego_trans, ego_rot = ego_pose['translation'], ego_pose['rotation']

    # --- 1. RUN INFERENCE ---
    visual_segs = get_reachable_lanes_inference(nmap, ego_trans, ego_rot)
    final_batches = get_triplet_batches(nmap, ego_trans, ego_rot, visual_segs)
    
    # --- 2. PRINT BATCHING CONSOLE OUTPUT ---
    print(f"Sample: {sample_token}")
    print("\n" + "="*50)
    print("=== TENSOR COLUMN LEGEND ===")
    print("Col 0: Lateral (m)   - Sideways distance")
    print("Col 1: Forward (m)   - Distance ahead")
    print("Col 2: Dir_Lat       - Unit vector X (heading)")
    print("Col 3: Dir_Fwd       - Unit vector Y (heading)")
    print("Col 4: Curvature     - Turn sharpness (-0.5 to 0.5)")
    print("Col 5: s (0.0-1.0)   - Progress along path")
    print("Col 6: Flag          - (Currently Unused)")
    print("="*50 + "\n")
    
    print(f"=== IDENTIFIED DRIVABLE PATHS (Cropped to {FORWARD_MAX}m) ===")
    
    encoded_features = []
    for i, (path, ids) in enumerate(final_batches):
        if len(path) > 3:
            mat = calculate_feature_matrix(path, target_points=FIXED_POINT_COUNT)
            if mat is not None:
                encoded_features.append(mat)
                chain_str = " -> ".join([t[:4] for t in ids])
                print(f"Batch {i}: [{chain_str}] (Length: {path[-1,1]:.1f}m)")

    if encoded_features:
        print(f"\nFinal Input Tensor Shape: {np.array(encoded_features).shape} (Batch, Points, Features)")
    else:
        print("No valid paths found.")

    # --- 3. RUN VISUALIZATIONS ---
    if visual_segs:
        display_full_inference(nmap, ego_trans, ego_rot, visual_segs)
    if final_batches:
        display_batch_results(nmap, ego_trans, ego_rot, final_batches, visual_segs)